# Neural Network — Multi-Layer Perceptron (MLP)

**Course**: CMOR 438 / INDE 577 — Data Science & Machine Learning  
**Dataset**: International Football Results (1872–present)  
**Author**: Neriah29  

---

## The Progression So Far

| Algorithm | Decision boundary | Learns from data | Probability output |
|---|---|---|---|
| Perceptron | Linear | Yes (crude) | No |
| Logistic Regression | Linear | Yes (gradient descent) | Yes |
| KNN | Non-linear | No (lazy) | Approximate |
| **Neural Network** | **Non-linear** | **Yes (backprop)** | **Yes** |

The Neural Network combines everything: it learns like Logistic Regression (gradient descent), handles non-linear patterns like KNN, and outputs calibrated probabilities. It's the most powerful algorithm we've built.

---

## Architecture

```
Input layer        Hidden layer 1     Hidden layer 2     Output
(7 features)  →   (64 neurons)    →  (32 neurons)    →  (1 probability)
               ReLU activations    ReLU activations    Sigmoid activation
```

### Why ReLU in hidden layers?

ReLU (Rectified Linear Unit) = max(0, z). It passes positive values through and blocks negative ones. Two reasons it's preferred over sigmoid in hidden layers:
1. **No vanishing gradient** — sigmoid squashes large values toward 0 or 1 where gradients are tiny, making deep networks hard to train. ReLU doesn't have this problem.
2. **Sparse activation** — not all neurons fire for every input, which makes the network more efficient and generalized.

### Why sigmoid only at the output?

We only need a probability at the final output. Using sigmoid everywhere would reintroduce the vanishing gradient problem.

---

## Backpropagation in One Paragraph

After a forward pass computes predictions, the loss is computed. The error signal starts at the output layer (prediction − true label) and flows backwards through each layer. At each layer, gradients are computed for that layer's weights, and then the error is pushed further back through the weights and the ReLU derivative. ReLU's derivative acts as a gate — active neurons (z > 0) pass the error through, inactive neurons block it. Once every weight has its gradient, all weights update simultaneously via gradient descent. This is one epoch.


---
## Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

import sys
sys.path.insert(0, '../../src')
from football_ml.supervised_learning.mlp import MLP

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
SEED = 42

---
## 1. Why Neural Networks? The XOR Problem

Before touching football data, let's demonstrate why hidden layers matter using the XOR problem — the classic example that cannot be solved by a single linear model.

In [ ]:
# XOR: output is 1 when inputs differ, 0 when they match
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]], dtype=float)
y_xor = np.array([0, 1, 1, 0])

# Try to solve with Logistic Regression (linear — should fail)
sys.path.insert(0, '../../src')
from football_ml.supervised_learning.logistic_regression import LogisticRegression

lr = LogisticRegression(learning_rate=0.5, n_epochs=5000).fit(X_xor, y_xor)
mlp = MLP(hidden_layer_sizes=(4,), learning_rate=0.5, n_epochs=5000, random_state=3).fit(X_xor, y_xor)

print(f'Logistic Regression accuracy on XOR: {lr.score(X_xor, y_xor):.0%}')
print(f'MLP accuracy on XOR:                 {mlp.score(X_xor, y_xor):.0%}')

In [ ]:
# Visualise the decision boundaries
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
xx, yy = np.meshgrid(np.linspace(-0.5, 1.5, 300), np.linspace(-0.5, 1.5, 300))
grid = np.c_[xx.ravel(), yy.ravel()]

for ax, model, title in zip(axes,
                             [lr, mlp],
                             ['Logistic Regression (linear — fails)', 'MLP (non-linear — solves it)']):
    Z = model.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdBu')
    ax.contour(xx, yy, Z, colors='black', linewidths=1.5)
    ax.scatter(X_xor[:, 0], X_xor[:, 1], c=y_xor, cmap='RdBu',
               s=200, edgecolors='black', linewidth=2, zorder=5)
    for i, (xi, label) in enumerate(zip(X_xor, y_xor)):
        ax.annotate(f'y={label}', xi + 0.05, fontsize=10)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlim(-0.5, 1.5)
    ax.set_ylim(-0.5, 1.5)

plt.suptitle('XOR Problem — Linear vs Non-linear Decision Boundary', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

Logistic Regression cannot draw a straight line that separates the XOR pattern. The MLP can — because its hidden layer learns to transform the input space into one where the classes become linearly separable.

---
## 2. Load & Engineer Features

In [ ]:
df = pd.read_csv('../../data/results.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)
df['home_win'] = (df['home_score'] > df['away_score']).astype(int)

def compute_team_rolling_stats(df, window=10):
    home_log = df[['date', 'home_team', 'home_score', 'away_score']].copy()
    home_log.columns = ['date', 'team', 'scored', 'conceded']
    away_log = df[['date', 'away_team', 'away_score', 'home_score']].copy()
    away_log.columns = ['date', 'team', 'scored', 'conceded']
    team_log = pd.concat([home_log, away_log]).sort_values('date').reset_index(drop=True)
    team_log['rolling_scored'] = (
        team_log.groupby('team')['scored']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    team_log['rolling_conceded'] = (
        team_log.groupby('team')['conceded']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )
    return team_log.drop_duplicates(subset=['date', 'team'], keep='last').set_index(['date', 'team'])

team_stats = compute_team_rolling_stats(df)

def get_stat(row, team_col, stat_col):
    try:
        return team_stats.loc[(row['date'], row[team_col]), stat_col]
    except KeyError:
        return np.nan

df['home_goals_rolling']    = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_scored'), axis=1)
df['home_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'home_team', 'rolling_conceded'), axis=1)
df['away_goals_rolling']    = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_scored'), axis=1)
df['away_conceded_rolling'] = df.apply(lambda r: get_stat(r, 'away_team', 'rolling_conceded'), axis=1)

home_wins = df.groupby('home_team').apply(lambda g: (g['home_score'] > g['away_score']).mean()).rename('home_win_rate')
away_wins = df.groupby('away_team').apply(lambda g: (g['away_score'] > g['home_score']).mean()).rename('away_win_rate')
df = df.join(home_wins, on='home_team').join(away_wins, on='away_team')
df['neutral'] = df['neutral'].astype(int)

feature_cols = [
    'home_goals_rolling', 'away_goals_rolling',
    'home_conceded_rolling', 'away_conceded_rolling',
    'home_win_rate', 'away_win_rate', 'neutral'
]
df_clean = df[feature_cols + ['home_win']].dropna()
print(f'Dataset: {df_clean.shape[0]} matches | Home win rate: {df_clean["home_win"].mean():.1%}')

---
## 3. Prepare Data

In [ ]:
X = df_clean[feature_cols].values
y = df_clean['home_win'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Train: {X_train_sc.shape} | Test: {X_test_sc.shape}')

---
## 4. Train the Model

In [ ]:
model = MLP(
    hidden_layer_sizes=(64, 32),
    learning_rate=0.01,
    n_epochs=1000,
    random_state=SEED
)
model.fit(X_train_sc, y_train)

print(f'Train accuracy: {model.score(X_train_sc, y_train):.3f}')
print(f'Test  accuracy: {model.score(X_test_sc,  y_test):.3f}')

---
## 5. Loss Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(model.loss_history_, color='#4878CF', linewidth=2)
ax.fill_between(range(len(model.loss_history_)), model.loss_history_, alpha=0.1, color='#4878CF')
ax.axhline(np.log(2), color='#E87272', linestyle='--', linewidth=1.2,
           label=f'Random baseline ({np.log(2):.3f})')
ax.set_xlabel('Epoch', fontsize=12)
ax.set_ylabel('Cross-Entropy Loss', fontsize=12)
ax.set_title('MLP Training Loss Curve', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## 6. Architecture Comparison

How does the number of hidden neurons affect performance?

In [ ]:
architectures = [
    (16,),
    (64,),
    (64, 32),
    (128, 64, 32),
]

results = []
for arch in architectures:
    m = MLP(hidden_layer_sizes=arch, learning_rate=0.01,
            n_epochs=500, random_state=SEED).fit(X_train_sc, y_train)
    results.append({
        'architecture': str(arch),
        'train_acc': m.score(X_train_sc, y_train),
        'test_acc':  m.score(X_test_sc,  y_test),
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(len(architectures))
ax.bar(x - 0.2, results_df['train_acc'], 0.35, label='Train', color='#4878CF')
ax.bar(x + 0.2, results_df['test_acc'],  0.35, label='Test',  color='#E87272')
ax.set_xticks(x)
ax.set_xticklabels(results_df['architecture'], fontsize=9)
ax.set_ylabel('Accuracy', fontsize=12)
ax.set_title('Architecture Comparison', fontsize=13, fontweight='bold')
ax.legend()
ax.set_ylim(0.5, 1.0)
plt.tight_layout()
plt.show()

### What to notice

Bigger networks have more capacity to learn — but more capacity also means more risk of **overfitting** (train accuracy much higher than test accuracy). The goal is the architecture where train and test accuracy are both high and close together.

---
## 7. Evaluation

In [ ]:
y_pred = model.predict(X_test_sc)
probs  = model.predict_proba(X_test_sc)

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Pred: 0', 'Pred: 1'],
            yticklabels=['True: 0', 'True: 1'], ax=axes[0])
axes[0].set_title('Confusion Matrix', fontsize=12, fontweight='bold')

# ROC curve
fpr, tpr, _ = roc_curve(y_test, probs)
roc_auc = auc(fpr, tpr)
axes[1].plot(fpr, tpr, color='#4878CF', linewidth=2, label=f'MLP (AUC={roc_auc:.3f})')
axes[1].plot([0,1],[0,1], 'gray', linestyle='--', linewidth=1)
axes[1].set_xlabel('False Positive Rate', fontsize=11)
axes[1].set_ylabel('True Positive Rate', fontsize=11)
axes[1].set_title('ROC Curve', fontsize=12, fontweight='bold')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

print(classification_report(y_test, y_pred, target_names=['Draw/Away', 'Home Win']))

---
## 8. Comparison with Previous Algorithms

In [ ]:
from football_ml.supervised_learning.logistic_regression import LogisticRegression
from football_ml.supervised_learning.knn import KNNClassifier

models = {
    'Logistic Regression': LogisticRegression(learning_rate=0.1, n_epochs=1000, random_state=SEED),
    'KNN (k=9)':           KNNClassifier(k=9),
    'MLP (64,32)':         MLP(hidden_layer_sizes=(64, 32), learning_rate=0.01, n_epochs=1000, random_state=SEED),
}

fig, ax = plt.subplots(figsize=(8, 5))
for name, m in models.items():
    m.fit(X_train_sc, y_train)
    fpr, tpr, _ = roc_curve(y_test, m.predict_proba(X_test_sc))
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={roc_auc:.3f})')

ax.plot([0,1],[0,1],'gray',linestyle='--',linewidth=1,label='Random')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve Comparison — All Classifiers', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()

---
## 9. Discussion — Does the MLP Suit Football?

### What it does well
- **Non-linear boundaries** — can capture complex patterns between features that no linear model could
- **Learns feature interactions** — hidden layers automatically combine features in useful ways
- **Scales well** — adding more neurons or layers increases capacity for complex datasets
- **Probability output** — smooth, continuous probabilities from the sigmoid output

### What it struggles with
1. **Black box** — unlike Logistic Regression, we can't read off which features matter from the weights. There are hundreds of weights across multiple layers.
2. **More hyperparameters** — architecture choice (layers, neurons), learning rate, epochs all need tuning.
3. **Needs more data** — more parameters means more data required to train well without overfitting.
4. **Football is still noisy** — even the most powerful model can't predict what a referee will call or when a star player gets injured.

### The ceiling

The MLP is the most powerful algorithm we've built — but on this football dataset the accuracy improvement over Logistic Regression may be modest. That's not a failure of the algorithm — it's a property of the data. Football is genuinely hard to predict from historical statistics alone.

---
## Summary

| | |
|---|---|
| **Algorithm type** | Supervised, binary classification |
| **Architecture** | Input → Hidden layers (ReLU) → Output (Sigmoid) |
| **Training** | Backpropagation + Gradient Descent |
| **Loss function** | Binary Cross-Entropy |
| **Key new concepts** | Backpropagation, ReLU, He initialization, architecture tuning |
| **Decision boundary** | Non-linear (arbitrary complexity with enough neurons) |
| **Football suitability** | Good — best so far, but limited by data noise |
